# Comprehension audio omni-modale - Qwen3-Omni

**Module :** 02-Audio-Advanced
**Niveau :** Avance
**Technologies :** Qwen3-Omni-30B-A3B-Instruct (transformers 4.57.6, torch 2.6 + CUDA 12.4), quantisation NF4 via bitsandbytes, `qwen-omni-utils`, librosa, soundfile, RTX 4090
**Duree estimee :** 30 minutes de manipulation, dont le chargement du modele (quelques minutes)
**VRAM :** 17,6 Go en 4 bits sur le device 2, mesure dans le carnet
**Environnement :** kernel `coursia-genai-audio` (conda env `coursia-genai-audio`, Python 3.13) - le carnet charge un vrai checkpoint et l'execute, il ne simule rien.

**Navigation :** [<< 02-12](02-12-SVS-Singing-Voice-RVC.ipynb) | 02-13 - LLM audio omni-modal Qwen2-Audio (en cours de revue) | 02-14 - Transcription audio Qwen3-ASR (en cours de revue) | [03-Orchestration >>](../03-Orchestration/README.md)

## Ce que ce carnet mesure

Le carnet 02-14 mesure l'axe **transcription** : des moteurs dedies a la reconnaissance vocale, sur un signal bruite. Il reste un axe non mesure : l'axe **omni-modal**, c'est-a-dire un modele unique qui ecoute de l'audio et repond en texte. C'est celui de la question de dialogue du carnet 02-13 : « que demande l'utilisateur a son assistant audio ? »

Ce carnet pose trois questions :

1. **Le modele tient-il ?** Le checkpoint public de la generation Qwen3 est un modele de **35,26 Md de parametres** : en precision `bfloat16` il pese **65,68 Gio**, et le device reserve a cette lane expose 23,99 Gio. Le carnet mesure donc ce qui se passe quand on le **quantifie a la volee** en 4 bits, et ce que cette quantification coute.
2. **Que comprend-il ?** Sur les memes extraits que le carnet 02-14, le modele transcrit-il et repond-il a la question posee ?
3. **Entend-il une difference de timbre ?** Les deux extraits portent **le meme contenu** avec deux timbres differents : la voix synthetisee d'origine, puis la meme phrase convertie par un modele de conversion de voix. Un modele qui n'ecouterait que le contenu decrirait la meme voix deux fois ; un modele qui ecoute le timbre les decrit differemment. C'est le test le plus simple pour separer les deux canaux.


In [1]:
# Parametres du carnet (injectables via Papermill)
seed = 42
MODEL_ID = "Qwen/Qwen3-Omni-30B-A3B-Instruct"
AUDIO_SOURCE = "assets/02-11-kokoro-source.wav"   # parole source (TTS Kokoro du gateway, carnet 02-5)
AUDIO_RVC = "assets/02-11-kokoro-rvc.wav"         # la meme phrase, timbre converti par RVC v2 (carnet 02-12)
CUDA_DEVICE_ORDER = "PCI_BUS_ID"
CUDA_DEVICE_INDEX = 2                             # device 2 reserve a cette lane (les devices 0 et 1 portent le vLLM de la flotte)
MAX_NEW_TOKENS = 256
QUANT_BITS = 4                                   # NF4 : le checkpoint bf16 ne tient pas sur le device
QUESTION_COMPRENHESION = ("Transcribe the speech in this audio, then say what the "
                           "speaker is asking about.")
QUESTION_TIMBRE = ("Describe the voice you hear: its timbre, apparent gender and age "
                    "of the speaker, and the recording quality.")


### Les parametres

Les deux entrees audio viennent du depot, dans `assets/` :

- `02-11-kokoro-source.wav` : la parole source, synthetisee par le TTS Kokoro du gateway multi-modeles (carnet 02-5), commise avec le carnet 02-12 ;
- `02-11-kokoro-rvc.wav` : **la meme parole** apres conversion de timbre RVC v2 (carnet 02-12).

Le couple est deliberement *meme contenu, autre voix* : c'est ce qui permet de separer deux questions. « Que dit la parole ? » doit etre stable d'un extrait a l'autre ; « quelle voix est-ce ? » doit les departager. Le carnet 02-14 a pose la premiere question sur ces memes fichiers ; celui-ci pose les deux.

Les deux questions posees au modele sont les memes pour les deux fichiers, et le decodage est **greedy** (`do_sample=False`) : la sortie est donc deterministe pour un etat de poids donne, ce qui rend la comparaison entre les deux fichiers lisible.

Le device 2 est fixe par le protocole de regie GPU de la lane : sur cette machine, les devices 0 et 1 portent le vLLM de la flotte et ne sont pas reservables. La cellule de setup **prouve** le device utilise en comparant son UUID a celui annonce par `nvidia-smi`.


In [2]:
# Setup : environnement, materiel, versions
# L'ordre compte : les variables CUDA sont posees AVANT le premier usage de torch,
# sinon c'est le premier device visible qui est choisi.
import io
import json
import os
import subprocess
import time
import warnings

os.environ["CUDA_DEVICE_ORDER"] = CUDA_DEVICE_ORDER
os.environ["CUDA_VISIBLE_DEVICES"] = str(CUDA_DEVICE_INDEX)

import librosa
import numpy as np
import soundfile as sf
import torch

warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")

# Inventaire des devices de la machine : la ligne du device 2 est celle que la lane reserve
tableau_gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,uuid,memory.used,memory.total,utilization.gpu",
     "--format=csv,noheader"],
    capture_output=True, text=True, encoding="utf-8", errors="replace", check=True).stdout.strip().splitlines()
print("--- inventaire GPU (nvidia-smi) ---")
for ligne in tableau_gpu:
    print("   ", ligne)

import transformers
import bitsandbytes
import qwen_omni_utils

print(f"\n--- versions ---")
print(f"torch            : {torch.__version__}")
print(f"transformers     : {transformers.__version__}")
print(f"bitsandbytes     : {bitsandbytes.__version__}")
print(f"cuda disponible  : {torch.cuda.is_available()}")
print(f"device visible   : {torch.cuda.get_device_name(0)}")
vram_total = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"VRAM visible     : {vram_total:.2f} Gio")
print(f"torch seed       : {torch.manual_seed(seed)}")

# Les deux extraits : duree, cadence, nombre de canaux
print(f"\n--- extraits audio ---")
for nom in (AUDIO_SOURCE, AUDIO_RVC):
    info = sf.info(nom)
    duree_s = info.frames / info.samplerate
    audio, cadence = librosa.load(nom, sr=None, mono=True)
    print(f"{nom} : {duree_s:.2f} s | {cadence} Hz | {info.channels} canal(x) | "
          f"amplitude rms {float(np.sqrt(np.mean(audio ** 2))):.5f}")


--- inventaire GPU (nvidia-smi) ---
    0, GPU-4100579c-0723-9ba3-0123-e9e439ec1636, 22847 MiB, 24564 MiB, 78 %
    1, GPU-c6b0177a-aa40-b1af-11fe-67084c48189c, 21950 MiB, 24564 MiB, 80 %
    2, GPU-b2d7d542-280b-79f8-2d92-56e57409784d, 148 MiB, 24564 MiB, 0 %



--- versions ---
torch            : 2.6.0+cu124
transformers     : 4.57.6
bitsandbytes     : 0.45.5
cuda disponible  : True
device visible   : NVIDIA GeForce RTX 4090
VRAM visible     : 23.99 Gio
torch seed       : <torch._C.Generator object at 0x0000016F3D8272F0>

--- extraits audio ---


assets/02-11-kokoro-source.wav : 4.90 s | 24000 Hz | 1 canal(x) | amplitude rms 0.04290
assets/02-11-kokoro-rvc.wav : 4.88 s | 40000 Hz | 1 canal(x) | amplitude rms 0.12937


### Pourquoi le modele doit etre quantifie

Le checkpoint de la generation Qwen3 est un modele **Mixture-of-Experts** de **35,26 Md de parametres** : chaque token n'active qu'une petite fraction des experts, mais **tous** les poids doivent etre resident en memoire. En `bfloat16` (deux octets par poids), cela represente **65,68 Gio** a telecharger.

Le device reserve a cette lane expose 23,99 Gio. L'arithmetique est donc sans appel : **le checkpoint ne tient pas en pleine precision** sur ce device, et il ne tiendrait pas non plus en `float16` ou en `int8` sur la plupart des configurations de GPU de bureau.

La voie mesuree ici est la **quantification a la volee** : `bitsandbytes` charge le checkpoint par couche, convertit chaque matrice de poids en NF4 (quatre bits, avec une double quantification) et calcule en `bfloat16`. Ce n'est pas une version « reduite » publiee par l'editeur : ce sont **les poids officiels**, convertis a la volee, et le carnet mesure explicitement le prix de cette conversion.

La contrepartie honnete : un modele quantifie n'est plus exactement le modele publie. Les reponses de ce carnet sont donc des reponses d'un modele **en 4 bits**, et cette condition de mesure fait partie du resultat.


In [3]:
# Mesure du checkpoint TEL QU'IL EST PUBLIE, avant tout chargement en memoire
# On interroge l'index safetensors : la taille reelle sur disque et le nombre de
# parametres, lus dans les en-tetes des shards (sans charger les poids).
from huggingface_hub import snapshot_download

OCTETS_PAR_DTYPE = {"BF16": 2, "F16": 2, "F32": 4, "F64": 8, "I64": 8,
                    "I32": 4, "I16": 2, "I8": 1, "U8": 1, "BOOL": 1}

dossier = snapshot_download(MODEL_ID, allow_patterns=["*.json", "*.safetensors.index.json"])
chemin_index = os.path.join(dossier, "model.safetensors.index.json")
index = json.load(io.open(chemin_index, encoding="utf-8"))
shards = sorted(set(index["weight_map"].values()))

taille_octets = sum(os.path.getsize(os.path.join(dossier, s)) for s in shards)
n_parametres = 0
n_tenseurs = 0
octets_poids = 0
for shard in shards:
    with open(os.path.join(dossier, shard), "rb") as f:
        taille_entete = int.from_bytes(f.read(8), "little")
        entete = json.loads(f.read(taille_entete))
    for nom_tenseur, info in entete.items():
        if nom_tenseur == "__metadata__":
            continue
        n_tenseurs += 1
        produit = 1
        for dim in info["shape"]:
            produit *= int(dim)
        n_parametres += produit
        octets_poids += produit * OCTETS_PAR_DTYPE.get(info["dtype"], 2)

print(f"shards            : {len(shards)}")
print(f"tenseurs          : {n_tenseurs}")
print(f"poids sur disque  : {taille_octets / 2**30:.2f} Gio")
print(f"parametres        : {n_parametres / 1e9:.2f} Md")
print(f"octets de poids   : {octets_poids / 2**30:.2f} Gio")
print(f"VRAM du device 2  : {vram_total:.2f} Gio")
print(f"le checkpoint tient en pleine precision ? "
      f"{'oui' if taille_octets / 2**30 <= vram_total else 'NON'}")


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

shards            : 15
tenseurs          : 28010
poids sur disque  : 65.68 Gio
parametres        : 35.26 Md
octets de poids   : 65.68 Gio
VRAM du device 2  : 23.99 Gio
le checkpoint tient en pleine precision ? NON


### Le chargement : ce que la quantification fait reellement

On charge maintenant le checkpoint en 4 bits sur le device 2. Trois grandeurs sont mesurees :

- le **temps de chargement** (**250,3 s**), qui reflecte la conversion de chaque shard au format NF4 a la volee ;
- la **VRAM allouee** (**17,60 Go**), qui doit rester sous la capacite du device ;
- le **nombre de parametres** tels que le modele les expose apres quantification (**17,85 Md**).

Le troisieme point merite un avertissement : apres quantification, `bitsandbytes` emballe les poids en NF4 (quatre bits utiles dans un octet, puis une double quantification). Le `numel()` des tenseurs du modele charge ne compte donc **plus les parametres reels** : il faut le nombre de la cellule precedente (**35,26 Md**) pour parler du modele, et la valeur du module charge (**17,85 Md**) pour parler de ce que la couche contient. Les deux sont affichees cote a cote.


In [4]:
# Chargement du checkpoint officiel, quantifie en NF4 sur le device 2
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3OmniMoeForConditionalGeneration
from qwen_omni_utils import process_mm_info

quantification = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

vram_avant = torch.cuda.memory_allocated(0) / 2**30
t0 = time.perf_counter()
modele = Qwen3OmniMoeForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantification,
    device_map={"": 0},
    dtype=torch.bfloat16,
    max_memory={0: f"{int(vram_total * 0.92)}GiB"},
)
t_chargement = time.perf_counter() - t0
torch.cuda.synchronize(0)

vram_apres = torch.cuda.memory_allocated(0) / 2**30
vram_reservee = torch.cuda.memory_reserved(0) / 2**30
n_parametres_module = sum(p.numel() for p in modele.parameters())

print(f"temps de chargement  : {t_chargement:.1f} s")
print(f"VRAM allouee         : {vram_apres:.2f} Go (reservee {vram_reservee:.2f} Go)")
print(f"parametres reels     : {n_parametres / 1e9:.2f} Md (checkpoint publie)")
print(f"parametres du module : {n_parametres_module / 1e9:.2f} Md (NF4, emballe)")
print(f"module de parole     : {'present' if getattr(modele, 'has_talker', False) else 'absent'} "
      f"(la sortie audio du modele n'est pas mesuree ici)")

processor = AutoProcessor.from_pretrained(MODEL_ID)
print("processor            :", type(processor).__name__)


Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_interleaved', 'mrope_section', 'interleaved'}


Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section', 'interleaved'}


Loading checkpoint shards:   0%|          | 0/15 [00:00<?, ?it/s]

temps de chargement  : 250.3 s
VRAM allouee         : 17.60 Go (reservee 28.86 Go)
parametres reels     : 35.26 Md (checkpoint publie)
parametres du module : 17.85 Md (NF4, emballe)
module de parole     : present (la sortie audio du modele n'est pas mesuree ici)


The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


processor            : Qwen3OmniMoeProcessor


### Le protocole d'ecoute

Le modele est multi-modal : on lui donne un **extrait audio** et une **question en texte**, il repond en texte. La fonction ci-dessous encadre l'appel, et toutes les reponses du carnet passent par elle : les deux fichiers, les deux questions, le meme decodage.

Deux details techniques qui ne sont pas negociables :

- le modele est multi-modal, son encodeur audio **et** son decipher de texte sont quantifies en NF4, dont le calcul se fait en `bfloat16`. Les entrees audio issues du preprocesseur sont en `float32` : sans conversion explicite, le calcul echoue sur un conflit de types. La fonction convertit donc les tenseurs flottants avant l'appel ;
- la sortie audio du modele (le module de parole) est **desactivee** explicitement : le carnet mesure l'ecoute et la reponse en texte, pas la synthese vocale.

Le temps de reponse est mesure a chaque appel : c'est une donnee du carnet, pas un detail de mise au point.


In [5]:
# La fonction d'ecoute, utilisee pour les quatre appels du carnet
def caste_bf16(entrees):
    """Les entrees audio du preprocesseur sont en float32 ; les couches
    quantifiees calculent en bfloat16. On convertit les seuls tenseurs flottants."""
    return {
        cle: (valeur.to(torch.bfloat16) if torch.is_tensor(valeur) and valeur.is_floating_point()
              else valeur)
        for cle, valeur in entrees.items()
    }


def demander(chemin_audio, question):
    """Poser une question sur un extrait ; renvoie (reponse, duree en secondes)."""
    messages = [{"role": "user", "content": [
        {"type": "audio", "audio": chemin_audio},
        {"type": "text", "text": question},
    ]}]
    texte = processor.apply_chat_template(
        conversations=messages, add_generation_prompt=True, return_tensors="pt")
    audios, images, videos = process_mm_info(
        conversations=messages, use_audio_in_video=True)
    entrees = processor(text=texte, audio=audios, images=images, videos=videos,
                        return_tensors="pt").to(modele.device)
    longueur_prompt = entrees["input_ids"].shape[1]
    entrees = caste_bf16(entrees)

    t1 = time.perf_counter()
    sortie = modele.generate(**entrees, max_new_tokens=MAX_NEW_TOKENS,
                             return_audio=False, do_sample=False)
    torch.cuda.synchronize(0)
    duree = time.perf_counter() - t1

    ids = sortie
    if isinstance(sortie, (tuple, list)):
        ids = sortie[0]
    elif hasattr(sortie, "sequences"):
        ids = sortie.sequences
    reponse = processor.batch_decode(
        ids[:, longueur_prompt:], skip_special_tokens=True)[0]
    return reponse.strip(), duree


print("fonction d'ecoute prete")


fonction d'ecoute prete


In [6]:
# Ecoute 1 : que dit la parole source ?
reponse_source_comp, t_source_comp = demander(AUDIO_SOURCE, QUESTION_COMPRENHESION)
print(f"=== {AUDIO_SOURCE} [comprehension] {t_source_comp:.1f} s")
print(reponse_source_comp)


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


=== assets/02-11-kokoro-source.wav [comprehension] 43.9 s
The speaker says: "By Nvnu Dan C module, can sacker a la conversion de watching t?"

The speaker is asking in French, "By the Nvnu Dan C module, can Sacker do the conversion of watching t?"


### Ecoute 1 : la lecture du contenu, sur la parole source

La reponse ne se presente pas comme une transcription brute : le modele met la phrase entendue **entre guillemets**, puis reformule l'intention de l'interlocuteur. Deux observations.

- **La structure est reconnue, la surface ne l'est pas.** Le modele retrouve l'ossature de la phrase - une question sur une conversion, dans un cadre de module - et signale lui-meme que le fragment est en francais. Mais les mots francais sortent translitteres en anglais : `By Nvnu Dan C module` pour une formule qui ressemble a « dans ce module », et la fin de phrase est restituee de facon approximative (`conversion de watching t`).
- **Le modele ne reecrit pas la phrase en francais.** Il prefere citer ce qu'il a cru entendre, y compris son etrangete, plutot que de produire une phrase française plausible. C'est un comportement de lecture, pas de reecriture.

Cette degradation n'est pas propre a ce moteur : le carnet 02-14 (PR #27925) a montre, sur le **meme fichier**, que trois moteurs d'annees differentes lisent le meme squelette et se separent sur la fin de phrase. La degradation est donc dans le signal, pas dans l'ecoute.


In [7]:
# Ecoute 2 : quelle voix est-ce ?
reponse_source_tim, t_source_tim = demander(AUDIO_SOURCE, QUESTION_TIMBRE)
print(f"=== {AUDIO_SOURCE} [timbre] {t_source_tim:.1f} s")
print(reponse_source_tim)


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


=== assets/02-11-kokoro-source.wav [timbre] 69.7 s
*   **Timbre:** The voice has a clear, mid-range, and somewhat formal timbre. It sounds like a standard, synthesized text-to-speech voice.
*   **Apparent Gender and Age:** The speaker sounds like a neutral, adult female voice.
*   **Recording Quality:** The recording quality is very high. The audio is clean, with no background noise, static, or distortion. The voice is perfectly clear and easy to understand.


### Ecoute 2 : la lecture du timbre, sur la parole source

La reponse est **structuree** (timbre, genre et age apparents, qualite d'enregistrement) et ses adjectifs ne sont pas degrades : la voix est decrite comme `clear, mid-range`, `formal`, une voix de synthese « standard », de genre et d'age `neutral, adult female`, avec une qualite d'enregistrement `very high`, sans bruit de fond ni distorsion.

C'est le constat central du carnet, et il se pose des la premiere paire d'ecoutes : **les deux canaux ne se comportent pas de la meme facon sur le meme signal**. L'axe du timbre repose sur des proprietes spectrales larges - bande passante, rapport de l'energie aux formants, bruit de fond, articulation - qui resistent a la degradation. L'axe du contenu repose sur des indices fins - consonnes, morphologie, ordre des mots - qui, eux, disparaissent. Un meme extrait audio peut donc etre parfaitement decrit par sa voix et illisible par son texte.


In [8]:
# Ecoute 3 : la lecture du contenu, sur la parole convertie
reponse_rvc_comp, t_rvc_comp = demander(AUDIO_RVC, QUESTION_COMPRENHESION)
print(f"=== {AUDIO_RVC} [comprehension] {t_rvc_comp:.1f} s")
print(reponse_rvc_comp)


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


=== assets/02-11-kokoro-rvc.wav [comprehension] 34.1 s
The speaker is asking about the conversion of "watching tea" into "sacré" in the context of the "module d'enseignement" (teaching module).


### Ecoute 3 : la lecture du contenu, sur la parole convertie

La lecture est elle aussi degradee, mais **pas la meme degradation** que sur la source. Le modele rend une phrase qui fait apparaitre des fragments francais que la source ne produisait pas : `sacré` (accentue) et `module d'enseignement` apparaissent explicitement, ou la version source donnait une translitteration (`Dan C module`) et une fin de phrase approximative.

Le decodage etant greedy, cet ecart n'est pas du hasard d'echantillonnage : a poids et question identiques, seule l'audio change. **La conversion de timbre modifie donc la lecture du contenu** sur ce signal. C'est le resultat qui contrarie l'intuition du carnet 02-14, qui attendait une stabilite du contenu d'un timbre a l'autre : ici, l'axe du contenu n'est pas stable, alors que l'axe du timbre - on va le voir - l'est.

Deux lectures possibles, et le carnet ne pretend pas trancher : soit la conversion degrade le signal au point de perdre de l'information fine, soit elle modifie les formants qui portent les consonnes. Une mesure supplementaire serait necessaire pour distinguer les deux ; ici, on se contente de constater l'ecart.


In [9]:
# Ecoute 4 : quelle voix est-ce, sur la parole convertie ?
reponse_rvc_tim, t_rvc_tim = demander(AUDIO_RVC, QUESTION_TIMBRE)
print(f"=== {AUDIO_RVC} [timbre] {t_rvc_tim:.1f} s")
print(reponse_rvc_tim)


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


=== assets/02-11-kokoro-rvc.wav [timbre] 88.3 s
*   **Timbre:** The voice has a synthetic, robotic, and monotonous timbre. It sounds like a text-to-speech (TTS) engine, with a very flat and artificial quality.
*   **Apparent Gender and Age:** The voice is clearly synthesized and does not have a discernible human gender. It sounds like a generic, ageless, computer-generated voice.
*   **Recording Quality:** The recording quality is very high. The audio is exceptionally clear, with no background noise, static, or distortion. The sound is clean and digital, which is characteristic of a direct software output.


### Ecoute 4 : la lecture du timbre, sur la parole convertie

La discrimination est nette et de bon sens : la version convertie est decrite comme `synthetic, robotic, monotonous`, avec une qualite `flat and artificial`, un genre « sans genre » (`generic, ageless, computer-generated`) et une qualite d'enregistrement `exceptionally clear`, `clean and digital`.

Contraste avec la source, decrite comme une voix neutre et adulte : **le meme modele, la meme question, deux fichiers au contenu identique, et deux descriptions de voix opposees**. C'est la preuve que l'axe du timbre est reellement mesure par le moteur, et qu'il n'est pas un remplissage generique : une reponse identique aux deux fichiers aurait montre l'inverse.

Le resultat est coherent avec le carnet 02-12, qui mesurait la conversion par la hauteur fondamentale et les ecarts de formants : la conversion remplace effectivement le timbre, et elle le remplace de facon detectable par un tiers.


In [10]:
# Synthese : comparer les quatre reponses sur des criteres explicites
def normaliser(texte):
    """Minuscules et ponctuation retirees, pour comparer du vocabulaire."""
    ponctuation = ".,;:!?'\"()[]{}"
    return [mot.strip(ponctuation).lower()
            for mot in texte.split() if mot.strip(ponctuation)]


def jaccard(a, b):
    """Recouvrement de vocabulaire entre deux reponses."""
    mots_a, mots_b = set(normaliser(a)), set(normaliser(b))
    if not (mots_a | mots_b):
        return 0.0
    return len(mots_a & mots_b) / len(mots_a | mots_b)


reponses = {
    ("source", "contenu"): reponse_source_comp,
    ("source", "timbre"): reponse_source_tim,
    ("rvc", "contenu"): reponse_rvc_comp,
    ("rvc", "timbre"): reponse_rvc_tim,
}
durees = {
    ("source", "contenu"): t_source_comp,
    ("source", "timbre"): t_source_tim,
    ("rvc", "contenu"): t_rvc_comp,
    ("rvc", "timbre"): t_rvc_tim,
}

print("--- recouvrement lexical entre les deux fichiers (Jaccard) ---")
print(f"axe du contenu : {jaccard(reponse_source_comp, reponse_rvc_comp):.3f}")
print(f"axe du timbre  : {jaccard(reponse_source_tim, reponse_rvc_tim):.3f}")

MARQUEURS = {
    "contenu": ("module", "conversion", "watching", "sacré"),
    "timbre": ("synthetic", "neutral", "female", "robotic", "ageless"),
}
print("\n--- marqueurs attendus, presents ou absents ---")
for (fichier, axe), texte in reponses.items():
    mots = set(normaliser(texte))
    trouves = [marqueur for marqueur in MARQUEURS[axe] if marqueur in mots]
    print(f"{fichier:7s} {axe:8s} : "
          f"{', '.join(trouves) if trouves else '(aucun)'}")

print("\n--- durees de reponse ---")
for (fichier, axe), duree in durees.items():
    print(f"{fichier:7s} {axe:8s} : {duree:6.1f} s")


--- recouvrement lexical entre les deux fichiers (Jaccard) ---
axe du contenu : 0.310
axe du timbre  : 0.471

--- marqueurs attendus, presents ou absents ---
source  contenu  : module, conversion, watching
source  timbre   : neutral, female
rvc     contenu  : module, conversion, watching, sacré
rvc     timbre   : synthetic, robotic, ageless

--- durees de reponse ---
source  contenu  :   43.9 s
source  timbre   :   69.7 s
rvc     contenu  :   34.1 s
rvc     timbre   :   88.3 s


### Ce que la comparaison etablit

La synthese dit en chiffres ce que les quatre lectures disaient en mots.

- **Axe du timbre** : le recouvrement de vocabulaire entre les deux fichiers est de **0,471**, et les jugements ports sont **opposes**. Les reponses partagent la meme grille descriptive (timbre, genre et age apparents, qualite d'enregistrement) et divergent sur le fond : voix neutre et adulte d'un cote, voix robotique et sans genre de l'autre. C'est la signature d'une reponse **discriminante** : meme format, verdict different. Les marqueurs attendus le confirment - `neutral` et `female` n'apparaissent que du cote source, `synthetic`, `robotic` et `ageless` que du cote converti.
- **Axe du contenu** : le recouvrement est plus faible (**0,310**). Les deux reponses partagent les mots d'ancrage (`module`, `conversion`) et divergent sur la fin de phrase - c'est exactement le point de separation que le carnet 02-14 avait observe entre trois moteurs d'annees differentes. Le contenu n'est pas stable d'un timbre a l'autre.

Les durees de reponse mesurent le cout de l'exercice : de **34,1 s** a **88,3 s** par appel. Chaque appel passe par l'encodeur audio puis par le decodeur de texte, sur un modele de plus de trente-cinq milliards de parametres dont la majorite est inactive a chaque token genere (architecture Mixture-of-Experts).


### Limites de la mesure

- **Le modele mesure est quantifie.** Les poids sont convertis en NF4 a la volee pour tenir dans la memoire du device. Ce n'est donc pas le checkpoint publie en `bfloat16` : la quantification peut amplifier les erreurs de transcription. Toute comparaison avec un moteur non quantifie (le carnet 02-14) porte sur des configurations differentes, et le carnet ne pretend pas le contraire.
- **Un couple, pas un echantillon.** Un seul extrait source et un seul timbre de conversion. Ce qui est etabli ici - un axe qui discriminate, un axe qui ne discrimine pas - vaut pour ce couple ; l'echantillon ne permet pas de generaliser a la parole humaine.
- **La sortie audio du modele n'est pas mesuree.** Le module de parole est desactive : le carnet mesure l'ecoute, pas la synthese. Le modele sait-faire les deux, mais le carnet ne le montre pas.
- **Le timbre est un jugement, pas une mesure.** Les descripteurs de l'axe du timbre sont produits par le modele lui-meme : ils sont discriminants, mais leur justesse n'est verifiee par aucun temoin externe dans ce carnet. Le carnet 02-12 fournit le point de comparaison (mesure de hauteur et de formants avant/apres), pas la validation du vocabulaire employe.
- **Les temps de reponse dependent du materiel.** Ils sont ceux d'un device 4090 en 4 bits ; ils ne se transposent ni a une autre carte ni a une autre version du checkpoint.


## Synthese

- **Un checkpoint de trente-cinq milliards de parametres sur un device de vingt-quatre Gio.** Le modele publie pese 65,68 Gio en `bfloat16` ; quantifie en NF4 a la volee, il se charge sur le device 2 en 250,3 s et occupe 17,60 Go de memoire allouee. C'est la mesure qui rend le reste possible, et elle est faite ici plutot que supposee.
- **Deux axes, deux comportements.** Sur le meme signal, l'axe du timbre est decrit avec justesse et **discrimine** les deux fichiers ; l'axe du contenu est degrade et **ne discrimine pas** les deux fichiers. Un moteur multi-modal ne se juge donc pas sur une reponse, mais sur la stabilite de chacune de ses lectures.
- **La degradation du signal est un fait mesure, pas une excuse.** Le carnet 02-14 a montre que trois moteurs d'annees differentes lisent le meme squelette sur ce fichier ; celui-ci confirme par un quatrieme moteur et par une autre voie d'appel - une question en texte plutot qu'une transcription.
- **La conversion de timbre se voit sur l'axe du timbre, pas sur l'axe du contenu.** C'est l'inverse de ce que l'on attendrait intuitivement, et c'est le resultat le plus utile du carnet pour qui veut choisir un moteur d'ecoute.
- **Ce que le carnet ne dit pas.** Il ne mesure pas la sortie audio du modele, il ne valide pas le vocabulaire du timbre par un temoin externe, et un couple audio ne fait pas une statistique.
